In [1]:
import mlflow
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
import numpy as np
from itertools import product
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from collections import defaultdict

import sys
sys.path.insert(0, "../../run")
from const import REPO_PATH
from experiment_config import TRAINGING_CONFIG

sys.path.insert(1, f"{REPO_PATH}")
from src.model.experiment_utils import *
from src.model.model_utils import *
from src.model.models import LSTMModel

In [2]:
sequence_length=5

In [3]:
features_path = f"{TRAINGING_CONFIG['features_path']}/all_combined_features.csv"
output_dir = f"{REPO_PATH}/models/ml_models"

In [4]:
seasons=sorted(TRAINGING_CONFIG['seasons'])
target_dfs=[pd.read_csv(f"{TRAINGING_CONFIG['processed_data_path']}/{season}/all_target_df.csv") for season in seasons[1:]]
for df in target_dfs:
    df['date']= pd.to_datetime(df['date'])
target_df = pd.concat(target_dfs, ignore_index=True)
target_columns= list(TRAINGING_CONFIG['target_ranges'].keys())

In [5]:
feature_df = pd.read_csv(features_path)
feature_df['date']= pd.to_datetime(feature_df['date'])
feature_df, target_df=align_on_keys(feature_df, target_df, TRAINGING_CONFIG['key_columns'])

In [6]:
model_param_grid = {
    'model_params.hidden_dim': [64],
    'model_params.num_layers': [1],
    'model_params.dropout': [0.2],
    'learning_rate': [0.001],
    'batch_size': [64],
    'epochs': [100],
    'device': ['auto'],
    'use_padding': [False],
}

In [7]:
run_multiclass_distribution_experiment_seq(
    feature_df=feature_df,
    target_df=target_df,
    target_ranges=TRAINGING_CONFIG['target_ranges'],
    model_wrapper_class=lambda **params: TorchSequenceWrapper(
        model_class=LSTMModel,
        model_params={k.replace('model_params.', ''): v for k, v in params.items() if k.startswith('model_params.')},
        sequence_length=sequence_length,
        date_col='date',
        home_col='home',
        away_col='away',
        use_padding=True,
        learning_rate=params.get('learning_rate', 0.001),
        batch_size=params.get('batch_size', 64),
        epochs=params.get('epochs', 100),
        device=params.get('device', 'auto'),
    ),
    model_param_grid=model_param_grid,
    test_size=0.2,
    experiment_name='sequence_lstm_experiment',
    repo_path=REPO_PATH,
    model_name='LSTM',
    target_columns=target_columns,
    k=5,
    key_columns=TRAINGING_CONFIG['key_columns'],   
)

param_grid: {'model_params.hidden_dim': [64], 'model_params.num_layers': [1], 'model_params.dropout': [0.2], 'learning_rate': [0.001], 'batch_size': [64], 'epochs': [100], 'device': ['auto'], 'use_padding': [False]}
having param combinations: [{'model_params.hidden_dim': 64, 'model_params.num_layers': 1, 'model_params.dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 64, 'epochs': 100, 'device': 'auto', 'use_padding': False}]
Training fold 1/5...


ValueError: Found input variables with inconsistent numbers of samples: [608, 1176]